# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TEJAS7356/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Suggested Improvements

1. **Growing vs. declining content:** Define the label using a clearly specified impression-measurement window. Validate the pattern on later time periods and account for differences in topic and search demand. This would help establish whether the observed relationship generalizes beyond the current sample.

2. **AI model performance:** Compare OpenAI and Gemini content within matched age and topic groups, using the same performance metrics and evaluation period. Report sample sizes and uncertainty for each group before making broader claims.

These improvements would strengthen the evidence while keeping the findings observational unless a suitable causal study is conducted.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### 2. My model under an honest split (before/after)

I evaluated my Decision Tree Classifier using two different validation strategies.

**Before — Random split**

The model achieved an F1 score of **0.425** using a random 80/20 train-test split. A random split may place content from the same client in both sets, which can make evaluation less representative of unseen clients.

**After — Client-grouped split**

I used `GroupShuffleSplit` with `client_hash_id` to ensure that clients in the training set did not appear in the test set. The model achieved an F1 score of **0.478**.

| Validation method | F1 score |
|---|---:|
| Random split (80/20) | 0.425 |
| Client-grouped split (80/20) | 0.478 |
| Difference | +0.053 |

**Interpretation**

The client-grouped split produced an F1 score 0.053 higher than the random split. This does not prove that grouped validation is generally better or that the model is more reliable. It shows that, on this particular split, the model achieved a higher F1 score when evaluated on unseen clients. Further validation with different random seeds and a carefully defined target is needed.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Week 5 model: random vs client-grouped validation

random_f1 = 0.425
grouped_f1 = 0.478

print("Random split F1:", random_f1)
print("Client-grouped split F1:", grouped_f1)
print("Difference:", round(grouped_f1 - random_f1, 3))

if grouped_f1 > random_f1:
    print("Grouped split produced a higher F1 score.")
elif grouped_f1 < random_f1:
    print("Grouped split produced a lower F1 score.")
else:
    print("Both splits produced the same F1 score.")

Random split F1: 0.425
Client-grouped split F1: 0.478
Difference: 0.053
Grouped split produced a higher F1 score.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

I audited the final feature set for possible data leakage.

1. **Future information leakage:** `mar_clicks` and `mar_impressions` were excluded from the features because they belong to March, the target month. Using them could reveal future information to the model.

2. **Identifier leakage:** `client_hash_id` and `content_hash_id` were excluded from the model features. The client ID was used only to create the grouped train-test split.

3. **Target leakage:** The target was created by comparing March clicks with February clicks. The model used only February impressions, clicks, and average position as input features.

4. **Unequal month lengths:** February has 28 days and March has 31 days. Comparing raw monthly click totals may incorrectly label content as declining. A future version should compare daily click rates or use equivalent time windows.

5. **Selection bias:** The inner join retained only content pairs present in both months. The results may not represent content missing from either month.

6. **Validation leakage:** The random split may place content from the same client in training and testing. The client-grouped split avoids this by keeping each client in only one set.

**Conclusion:** The final feature set avoids direct use of March outcome columns and client identifiers as predictors. However, the target definition and matched-pair selection may still introduce limitations. The F1 scores should be treated as preliminary results, not proof of reliable future predictions.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
features = [
    "feb_impressions",
    "feb_clicks",
    "feb_avg_position"
]

future_columns = ["mar_clicks", "mar_impressions"]
id_columns = ["client_hash_id", "content_hash_id"]

print("Final model features:", features)
print("Future columns excluded:", future_columns)
print("ID columns excluded from features:", id_columns)

assert not any(col in features for col in future_columns)
assert not any(col in features for col in id_columns)

print("Leakage check passed.")

Final model features: ['feb_impressions', 'feb_clicks', 'feb_avg_position']
Future columns excluded: ['mar_clicks', 'mar_impressions']
ID columns excluded from features: ['client_hash_id', 'content_hash_id']
Leakage check passed.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

My Decision Tree Classifier uses February search impressions, clicks, and average position to identify content that may need performance review.

The model achieved an F1 score of 0.425 with a random split and 0.478 with a client-grouped split. The grouped split produced a 0.053 higher F1 score in this evaluation, but this result alone does not prove that the model generalizes better in all cases.

The target compares raw February and March click totals, although the months have different lengths. Natural traffic fluctuations and the use of matched content pairs may also affect the results.

Therefore, the model should be treated as a decision-support tool for prioritizing content for review, not as proof that content is declining or that updating it will improve performance. Further validation with a comparable target and additional time periods is needed.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.